# Filter airports = (leave only EU airoports)
We use [Open airoports](https://ourairports.com/data/) dataset



In [1]:
import pandas as pd
df = pd.read_csv("./data/raw/airports.csv")
df.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [2]:
df = df[['icao_code', 'name', 'type', 'iso_country', 'iso_region', 'municipality', 'latitude_deg', 'longitude_deg']]

from graph_creation.config import EU_COUNTRY_CODES
only_eu_mask = df['iso_country'].apply(lambda x: x in EU_COUNTRY_CODES)
only_aeroports_mask = df['type'].apply(lambda x: x in ['small_airport', 'medium_airport', 'large_airport'])
icao_code_exists_mask = ~df['icao_code'].isna()

mask = only_eu_mask & only_aeroports_mask & icao_code_exists_mask

eu_aeroports_df = df[mask]
eu_aeroports_df = eu_aeroports_df[~eu_aeroports_df['municipality'].isna()]
eu_aeroports_df.head()

,icao_code,name,type,iso_country,iso_region,municipality,latitude_deg,longitude_deg
12527,EBLB,Elsenborn Air Base,small_airport,BE,BE-WLG,Bütgenbach,50.484818,6.183014
12580,LBDM,Dve Mogili Airfield,small_airport,BG,BG-18,Dve Mogili,43.606541,25.890690
12592,LBBB,Belchinski Bani Airstrip,small_airport,BG,BG-23,Belchinski Bani,42.374713,23.392548
12594,LBBL,Blagoevo Airfield,small_airport,BG,BG-17,Blagoevo,43.455488,26.431655
12597,LBDR,Draganovtsi Airfield,small_airport,BG,BG-07,Draganovtsi,42.943042,25.169528


In [3]:
eu_aeroports_df.shape

(1178, 8)

In [4]:
eu_aeroports_df.isna().sum().sum()

np.int64(0)

In [5]:
eu_aeroports_df.to_csv("./data/raw/eu_aeroports.csv", index=False)

## Aeroports to municipality aggregation
Several municipalities have more than one airport, lets aggregate them to municipality unit

In [6]:
# cheating a bit :) 
# we do not use in the graph. This data should be learned from the graph itself
eu_aeroports_df['type'].value_counts()

type
small_airport     602
medium_airport    378
large_airport     198
Name: count, dtype: int64

In [7]:
# the same as eu_aeroports_df[['municipality']].value_counts().value_counts()
eu_aeroports_df.groupby('municipality')['icao_code'].count().value_counts()

icao_code
1    1128
2      22
3       2
Name: count, dtype: int64

In [8]:
eu_aeroports_df[['municipality', 'iso_country']].value_counts().value_counts()

count
1    1130
2      21
3       2
Name: count, dtype: int64

In [9]:
eu_aeroports_df[eu_aeroports_df['municipality'] == 'Mora']

,icao_code,name,type,iso_country,iso_region,municipality,latitude_deg,longitude_deg
25073,ESKM,Mora Airport,medium_airport,SE,SE-W,Mora,60.957901,14.51140
44086,LPMO,Morargil Airfield,small_airport,PT,PT-07,Mora,38.993532,-8.14201


In [10]:
municipality_df = (
    eu_aeroports_df
    .groupby(["municipality", "iso_country"], as_index=False)
    .agg(
        icao_codes=("icao_code", list),
        iso_country=("iso_country", "first"),
        iso_region=("iso_region", "first"),
        latitude_deg=("latitude_deg", "mean"),
        longitude_deg=("longitude_deg", "mean"),
    )
)
municipality_df.head()

,municipality,icao_codes,iso_country,iso_region,latitude_deg,longitude_deg
0,Aachen,[EDKA],DE,DE-NW,50.823055,6.186389
1,Aalborg,[EKYT],DK,DK-81,57.094763,9.849930
2,Aarhus,[EKAH],DK,DK-82,56.303331,10.618286
3,Abbeyshrule,[EIAB],IE,IE-LD,53.591373,-7.642947
4,Agen,[LFBA],FR,FR-NAQ,44.174171,0.593090


In [11]:
municipality_df['icao_codes'] = municipality_df['icao_codes'].apply(lambda x: ", ".join(x))

In [12]:
municipality_df[municipality_df['municipality'] == 'Mora']

,municipality,icao_codes,iso_country,iso_region,latitude_deg,longitude_deg
675,Mora,LPMO,PT,PT-07,38.993532,-8.14201
676,Mora,ESKM,SE,SE-W,60.957901,14.51140


In [13]:
municipality_df.rename({"icao_codes": "airports"}, inplace=True, axis=1)

In [14]:
municipality_df.to_csv("./data/municipality.csv", index=False)

# Graph creation
We load data about flight frome OpenSKY provider. We collect data for one week: from 16 feb (mon) til 22 feb (sun). We can not load the whole week (400 error - too big timeframe), so we create one graph for 1 or 2 days, and then we merge them into one graph for the week.

In [1]:
from graph_creation.bootstrap import bootsrap_create_save_graph_use_case
u = bootsrap_create_save_graph_use_case()

u.run("2026-02-16", day_interval_number=1, graph_name="2026-02-16")

loaded 108842 flights from OpenSky
flights with empty arr/dep = 33876
flights with empty icao or callsign = 39
unknown or non-eu departures: 63847
unknown or non-eu arrivals: 3918
self loops (flights within the same municipality): 654
Graph created with 406 nodes and 3145 edges, flights: 6508


In [ ]:
u.run("2026-02-17", day_interval_number=2, graph_name="2026-02-17_18")

loaded 225192 flights from OpenSky
flights with empty arr/dep = 70663
flights with empty icao or callsign = 108
unknown or non-eu departures: 134320
unknown or non-eu arrivals: 6890
self loops (flights within the same municipality): 1757
Graph created with 569 nodes and 3976 edges, flights: 11454


In [1]:
from graph_creation.bootstrap import bootsrap_create_save_graph_use_case
u = bootsrap_create_save_graph_use_case()

u.run("2026-02-19", day_interval_number=2, graph_name="2026-02-19_20")

loaded 234369 flights from OpenSky
flights with empty arr/dep = 73382
flights with empty icao or callsign = 142
unknown or non-eu departures: 138246
unknown or non-eu arrivals: 7961
self loops (flights within the same municipality): 1715
Graph created with 584 nodes and 4524 edges, flights: 12923


In [2]:
u.run("2026-02-21", day_interval_number=2, graph_name="2026-02-21_22")

loaded 218753 flights from OpenSky
flights with empty arr/dep = 69198
flights with empty icao or callsign = 113
unknown or non-eu departures: 128522
unknown or non-eu arrivals: 7725
self loops (flights within the same municipality): 1065
Graph created with 487 nodes and 4188 edges, flights: 12130


In [4]:
from graph_creation.adapters.input import GraphJsonLoader
g1 = GraphJsonLoader().load("./data/graphs_json_html/2026-02-16.json")
g2 = GraphJsonLoader().load("./data/graphs_json_html/2026-02-17_18.json")
g3 = GraphJsonLoader().load("./data/graphs_json_html/2026-02-19_20.json")
g4 = GraphJsonLoader().load("./data/graphs_json_html/2026-02-21_22.json")

In [6]:
week_g = g1 + g2 + g3 + g4
week_g

Graph(nodes=762, edges=7932, unknown_or_non_eu_dep=464935, unknown_or_non_eu_arr=26494, begin=2026-02-16 00:00:00 UTC, end=2026-02-22 23:59:59 UTC)

In [14]:
from graph_creation.adapters.output.graph import GraphPandasAdapter
graph_to_pandas = GraphPandasAdapter(week_g)

nodes_df = graph_to_pandas.nodes_to_df()
nodes_df.head()

,id,name,iso_country,iso_region,latitude,longitude,airports_count,out_flights_number,in_flights_number,nut3_code
0,"Lorient/Lann/Bihoué, FR",Lorient/Lann/Bihoué,FR,FR-BRE,47.760601,-3.440000,1,6,7,None
1,"Colombier-Saugnieu, Rhône, FR","Colombier-Saugnieu, Rhône",FR,FR-ARA,45.725996,5.090139,1,448,445,None
2,"Leck, DE",Leck,DE,DE-SH,54.790905,8.963439,1,9,7,None
3,"Melilla, ES",Melilla,ES,ES-ML,35.279800,-2.956260,1,27,39,None
4,"Hradec Králové, CZ",Hradec Králové,CZ,CZ-KR,50.253201,15.845200,1,16,16,None


In [ ]:
nodes_df.isna().sum() # nut3_code has not been set yet, it should be null

id                      0
name                    0
iso_country             0
iso_region              0
latitude                0
longitude               0
airports_count          0
out_flights_number      0
in_flights_number       0
nut3_code             762
dtype: int64

In [ ]:
nodes_df['id'].value_counts().value_counts() # excellent (＊◕ᴗ◕＊)

count
1    762
Name: count, dtype: int64

In [21]:
nodes_df.to_csv("./data/graph_csv/nodes.csv", index=False)

In [18]:
edges_df = graph_to_pandas.edges_to_df()
edges_df.head()

,from_id,to_id,weight,distance
0,"Aachen, DE","Köln (Cologne), DE",2,67.32
1,"Aachen, DE","Mönchengladbach, DE",1,50.45
2,"Aalborg, DK","Billund, DK",2,156.52
3,"Aalborg, DK","Copenhagen, DK",84,238.40
4,"Aalborg, DK","Amsterdam, NL",28,624.13


In [22]:
edges_df.isna().sum().sum()

np.int64(0)

In [23]:
edges_df.to_csv("./data/graph_csv/edges.csv", index=False)

In [38]:
nodes_df.to_csv("./data/graph_csv/nodes.csv", index=False)

In [24]:
flights_df = graph_to_pandas.flights_to_df()
flights_df.head()

,icao,callsign,count
0,3e1685,CHX3,8
1,3de0bf,DHFJB,2
2,45d069,VKG4587,1
3,45884d,OYBBM,4
4,4ac9f4,SAS1228,1


In [26]:
flights_df.isna().sum().sum()

np.int64(0)

In [27]:
flights_df.to_csv("./data/graph_csv/flights.csv", index=False)

In [28]:
from graph_creation.adapters.output import GraphJsonExporter

exporter = GraphJsonExporter("data")
exporter.export(week_g, "flights_week_graph.json")

# Get LAU to NUT3 df
## Parse EU NUT3 regions excel file
Correspondence tables, NUTS regions (EU-27), [eurostat](https://ec.europa.eu/eurostat/web/nuts/correspondence-tables)


In [7]:
import pandas as pd 

input_path = "./data/raw/eu_27_lau_nuts_2024.xlsx"
xls = pd.ExcelFile(input_path)

country_sheets = [e for e in xls.sheet_names if len(e) == 2]
print(len(country_sheets))

32


In [8]:
from graph_creation.config import EU_COUNTRY_CODES
# Greece is clalled Elada 'EL' instead of "GR" 
set(country_sheets) - set(EU_COUNTRY_CODES) 

{'CH', 'EL', 'LI', 'MK', 'NO', 'TR'}

In [11]:
my_countries_sheets = EU_COUNTRY_CODES.copy()
my_countries_sheets.remove("GR")
my_countries_sheets.append("EL")

In [ ]:
# TODO rename nut3 to nuts3 everywhere

def load_country_dfs(xls: pd.ExcelFile, country_sheets: list[str]) -> dict[str, pd.DataFrame]:
    country_dfs = {}

    for sheet in country_sheets:
        df = pd.read_excel(xls, sheet_name=sheet)

        columns_renaming = {"NUTS3": "nut3_code", "LAU NAME LATIN": "lau_name_latin"}
        df.rename(columns=columns_renaming, inplace=True)

        key = sheet if sheet != "EL" else "GR"
        country_dfs[key] = df[list(columns_renaming.values())].copy()

    return country_dfs

country_dfs = load_country_dfs(xls, my_countries_sheets)
country_dfs['GR'].head()

,nut3_code,lau_name_latin
0,EL301,Municipal Commune of Amaroussio (psevdo)
1,EL301,Municipal Commune of Aghia Paraskevi (psevdo)
2,EL301,Municipal Commune of Vrilissia (psevdo)
3,EL301,Municipal Commune of Iraklio (psevdo)
4,EL301,Municipal Commune of Kifissia


# Load nuts3 PPS data from eurostat
Gross domestic product (GDP) at current market prices by NUTS 3 region, [eurostat](https://ec.europa.eu/eurostat/databrowser/view/nama_10r_3gdp__custom_16402140/bookmark/table?lang=en&bookmarkId=3edea12e-d6ab-46cd-ab3e-d547d9bae3c1&c=1745826708191). We can load data with eurostat API. 

Our final table contains 3 columns:
- nut3_code
- pps, in millions of points
- pps per inhabitant, points

In [1]:
from graph_creation.adapters.input.pps_from_eurostat_loader import load_nuts3_gdp
pps_df = load_nuts3_gdp()
pps_df.head()

unit,nut3_code,pps,pps_per_inhabitant
0,AT,436203.37,47500.0
1,AT1,188494.63,46600.0
2,AT11,10384.18,34400.0
3,AT12,67871.36,39300.0
4,AT13,110239.09,54600.0


In [27]:
pps_df.to_csv("./data/nuts3_pps.csv", index=False)

# Get company from callsign


In [29]:
from graph_creation.adapters.input import GraphJsonLoader
from graph_creation.adapters.output.graph.pandas_adapter import GraphPandasAdapter
g = GraphJsonLoader().load("./data/flights_week_graph.json")
ga = GraphPandasAdapter(g)

flights_df = ga.flights_to_df()
flights_df.head()

,icao,callsign,count
0,3e1685,CHX3,8
1,3de0bf,DHFJB,2
2,45d069,VKG4587,1
3,45884d,OYBBM,4
4,4ac9f4,SAS1228,1


In [30]:
flights_df['callsign'].unique()

<StringArray>
[   'CHX3',   'DHFJB', 'VKG4587',   'OYBBM', 'SAS1228', 'SAS1204',  'NSZ9BH',
 'NSZ3079', 'SAS1218', 'SAS1214',
 ...
  'RYR3VZ', 'RYR7305',  'WMT7GQ', 'NJE114C', 'EVE7211',  'JAF96Y', 'FYL11VC',
 'NJE301Y', 'NJE474V', 'EUW5434']
Length: 13539, dtype: str

# Get plane from icao

In [31]:
flights_df['icao'].unique()

<StringArray>
['3e1685', '3de0bf', '45d069', '45884d', '4ac9f4', '4ac9f3', '4aca15',
 '4ac9f2', '4ac9f0', '4aca4f',
 ...
 '48b70c', '4b8da4', '503cf5', '4866f1', '3008df', '494116', '49d663',
 '48af0d', '49d62d', '408026']
Length: 5102, dtype: str